In [1]:
import os
from pyspark.sql import SparkSession

# 1. Força o encerramento da sessão atual para limpar o cache do Jupyter
try:
    spark.stop()
    print("Sessão antiga do Spark encerrada.")
except:
    pass

# 2. Configura as variáveis de caminho de forma dinâmica
try:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
except NameError:
    current_dir = os.getcwd()
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, '..')) if current_dir.endswith('src') else current_dir

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Define o warehouse do ICEBERG (onde seus dados reais vão morar)
warehouse_path = os.path.join(PROJECT_ROOT, "lakehouse")

# Define o warehouse PADRÃO do Spark (trava as pastas genéricas na pasta metadata)
spark_default_warehouse = os.path.join(PROJECT_ROOT, "metadata", "spark_warehouse")

# 3. Cria a sessão blindada com TODAS as configurações
spark = SparkSession.builder.appName("Lakehouse_Iceberg_Spark") \
    .config("spark.sql.warehouse.dir", spark_default_warehouse) \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", warehouse_path) \
    .getOrCreate()

print("Nova sessão iniciada. Catálogo padrão travado em: metadata/spark_warehouse")

26/03/18 12:36:45 WARN Utils: Your hostname, saiti-HP-Elite-SFF-600-G9-Desktop-PC resolves to a loopback address: 127.0.1.1; using 10.67.121.217 instead (on interface eno1)
26/03/18 12:36:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/silvio.ferreira/.ivy2/cache
The jars for the packages stored in: /home/silvio.ferreira/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-053cf67f-cc1f-4fef-9220-f6f47acef60f;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central


:: loading settings :: url = jar:file:/home/silvio.ferreira/Projetos/estudo_lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


:: resolution report :: resolve 63ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-053cf67f-cc1f-4fef-9220-f6f47acef60f
	confs: [default]
	0 artifacts copied, 1 already retrieved (0kB/2ms)
26/03/18 12:36:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setL

Nova sessão iniciada. Catálogo padrão travado em: metadata/spark_warehouse


In [3]:
# 4. Garante o namespace e define o caminho do Parquet bruto
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bronze")
caminho_parquet = os.path.join(PROJECT_ROOT, "lakehouse", "bronze", "events_raw", "events_raw.parquet")

# 5. Executa a criação da tabela
comando_sql = f"""
CREATE OR REPLACE TABLE iceberg.bronze.events_iceberg 
USING iceberg 
AS SELECT * FROM parquet.`{caminho_parquet}`;
"""

spark.sql(comando_sql)

print("\nTabela criada com sucesso!")
spark.sql("SELECT * FROM iceberg.bronze.events_iceberg").show(5)


Tabela criada com sucesso!
+---+-----+-------------------+
| id| name|           event_ts|
+---+-----+-------------------+
|  1|Alice|2025-01-01T10:00:00|
|  2|  Bob|2025-01-01T10:05:00|
+---+-----+-------------------+



In [ ]:
#REALIZANDO UM SELECT SIMPLES NA TABELA EVENTS_ICEBERG

comando_sql = f"""
SELECT * FROM iceberg.bronze.events_iceberg
"""

spark.sql(comando_sql).show(5)


+---+-----+-------------------+
| id| name|           event_ts|
+---+-----+-------------------+
|  1|Alice|2025-01-01T10:00:00|
|  2|  Bob|2025-01-01T10:05:00|
+---+-----+-------------------+

